# Supervised Learning mit dem Advertising-Datensatz

Dieses Notebook zeigt einen einfachen **Supervised-Learning-Workflow** ohne dass Sie den Algorithmus selbst programmieren müssen:

1. CSV-Datei hochladen
2. Zielvariable festlegen
3. Trainings- und Testdaten aufteilen
4. Lineare Regression trainieren
5. Vorhersagen auf Testdaten erzeugen
6. Modellgüte bewerten


In [ ]:
# 1. Datei hochladen
from google.colab import files
uploaded = files.upload()


In [ ]:
# 2. Daten einlesen und ansehen
import io
import pandas as pd

filename = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[filename]))

# typische Index-Spalte entfernen, falls vorhanden
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

print('Datensatz:', filename)
print('Zeilen und Spalten:', df.shape)
display(df.head())


## Zielvariable und Features

Beim Advertising-Datensatz ist **sales** die Zielvariable $y$.
Die Eingabevariablen $X$ sind **TV, radio und newspaper**.

In [ ]:
# 3. Zielvariable und Features festlegen
TARGET = 'sales'

X = df.drop(columns=[TARGET])
y = df[TARGET]

print('Features X:', list(X.columns))
print('Zielvariable y:', TARGET)


In [ ]:
# 4. Daten in Training (80 %) und Test (20 %) aufteilen
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print('Training:', len(X_train), 'Beobachtungen')
print('Test:', len(X_test), 'Beobachtungen')


In [ ]:
# 5. Lineare Regression trainieren
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

print('Intercept:', round(model.intercept_, 4))
print('\nKoeffizienten:')
for feature, coef in zip(X.columns, model.coef_):
    print(f'{feature}: {coef:.4f}')


In [ ]:
# 6. Vorhersagen für bisher nicht verwendete Testdaten
y_pred = model.predict(X_test)

results = X_test.copy()
results['sales_tatsaechlich'] = y_test.values
results['sales_vorhergesagt'] = y_pred
results['fehler'] = results['sales_tatsaechlich'] - results['sales_vorhergesagt']

display(results.head(10).round(2))


In [ ]:
# 7. Modellgüte bewerten
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'R²   = {r2:.3f}')
print(f'MAE  = {mae:.3f}')
print(f'RMSE = {rmse:.3f}')


**Interpretation:**

- **R²**: Welcher Anteil der Streuung von `sales` wird vom Modell erklärt?
- **MAE**: durchschnittlicher absoluter Vorhersagefehler
- **RMSE**: bestraft größere Vorhersagefehler stärker


In [ ]:
# 8. Tatsächliche und vorhergesagte Werte vergleichen
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred)
plt.xlabel('Tatsächliche Sales-Werte')
plt.ylabel('Vorhergesagte Sales-Werte')
plt.title('Tatsächlich vs. vorhergesagt')

min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val])
plt.show()


## Reflexionsfragen

1. Was sind in diesem Beispiel die **Features X** und die **Zielvariable y**?
2. Warum handelt es sich um **Supervised Learning**?
3. Warum ist dies ein **Regressionsproblem**?
4. Warum trennen wir Trainings- und Testdaten?
5. Welche Variable hat den größten positiven Koeffizienten?
6. Wie gut sagt das Modell neue, bisher nicht zum Training verwendete Beobachtungen voraus?